# 09 Stage 2 Logistic Regression — Health Outcome Prediction

**Owner:** PBC  
**Targets:** `target_unmet_fp` (and `target_anc_gap` when m14 is available)  
**Depends on:** `07_data_integration.ipynb`, `08_clustering.ipynb`

In [ ]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.models.stage2_logistic import configure_logging, train_all_stage2_logistic
from src.models.stage2_xgboost import STAGE2_TARGETS, TARGET_DISPLAY, load_stage2_data

configure_logging()

In [ ]:
# Ensure Stage 2 inputs exist
stage2_files = [
    PROJECT_ROOT / 'data/processed/stage2/X_stage2_preclustering.csv',
    PROJECT_ROOT / 'data/processed/stage2/y_stage2_targets.csv',
    PROJECT_ROOT / 'outputs/stage2_results/cluster_assignments.csv',
]
if not all(p.exists() for p in stage2_files):
    raise FileNotFoundError('Run scripts/run_stage2_data_prep.py or 08_clustering.ipynb first.')
print('Stage 2 inputs found.')

In [ ]:
X_full, y = load_stage2_data()
print(f'Feature matrix (with cluster dummies): {X_full.shape}')
for col in y.columns:
    nn = y[col].notna().sum()
    pos = y.loc[y[col].notna(), col].sum() if nn else 0
    print(f'{col}: non-null N = {nn:,} | positive = {int(pos):,}')

In [ ]:
# Train separate LogisticRegression models per target (Section 6.2)
results = train_all_stage2_logistic()
metrics_df = results.get('metrics_df')
if metrics_df is not None:
    display_cols = ['Target', 'TrainSize', 'TestSize', 'ROC-AUC', 'F1-Score', 'CV_ROC-AUC', 'Barrier_Uplift']
    metrics_df[[c for c in display_cols if c in metrics_df.columns]]

In [ ]:
# Top odds-ratio predictors per target
for target_col in STAGE2_TARGETS:
    if target_col not in results.get('targets', {}):
        print(f'Skipped {TARGET_DISPLAY.get(target_col, target_col)} (no analytic sample)')
        continue
    coefs = results['targets'][target_col]['coefficients']
    print(f'\n=== {TARGET_DISPLAY.get(target_col, target_col)} — top odds ratios ===')
    print(coefs.head(10).to_string(index=False))